In [1]:
import sys
from pathlib import Path
import numpy as np

# Add project root to path
sys.path.append(str(Path("..").resolve()))


from helpers import Model, ModelData

# needed filepaths
from constants import PROCESSED_DATA_FOLDER, FITTED_MODEL_FOLDER

/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_data = ModelData.from_pickle(PROCESSED_DATA_FOLDER / "processed_data.pkl")

eval_indices = model_data.eval_indices

# Analyse der Modellvariablen (Summary Statistics)


In [3]:
########### Summary Statistics ###################
import pandas as pd

summary_data = {"Variable": [], "Mean": [], "SD": []}

# Rating level
summary_data["Variable"].append("Rating")
summary_data["Mean"].append(np.mean(model_data.ratings))
summary_data["SD"].append(np.std(model_data.ratings, ddof=1))

# Berechne Zeitdifferenzen zwischen Bewertungen
temp_diff = np.diff(model_data.days)
temp_diff = temp_diff[temp_diff >= 0]  # Negative Werte entfernen
summary_data["Variable"].append("Days Between Ratings")
summary_data["Mean"].append(np.mean(temp_diff))
summary_data["SD"].append(np.std(temp_diff, ddof=1))

# Sentiment statistics
summary_data["Variable"].append("Sentiment")
summary_data["Mean"].append(np.nanmean(model_data.sentiment))
summary_data["SD"].append(np.nanstd(model_data.sentiment, ddof=1))

# Restaurant Density
summary_data["Variable"].append("Density")
summary_data["Mean"].append(np.mean(model_data.business_covariates["density"]))
summary_data["SD"].append(np.std(model_data.business_covariates["density"], ddof=1))

# Age (in Monaten, daher / 28)
summary_data["Variable"].append("Age (in months)")
summary_data["Mean"].append(np.mean(model_data.business_covariates["Age"]) / 28)
summary_data["SD"].append(np.std(model_data.business_covariates["Age"] / 28, ddof=1))

# Checkin
summary_data["Variable"].append("Check-in rate")
summary_data["Mean"].append(np.mean(model_data.business_covariates["Checkin"]))
summary_data["SD"].append(np.std(model_data.business_covariates["Checkin"], ddof=1))

# Chain status
summary_data["Variable"].append("Chain status")
summary_data["Mean"].append(np.mean(model_data.business_covariates["chain"]))
summary_data["SD"].append(np.std(model_data.business_covariates["chain"], ddof=1))

# ZRI (Rent Level)
summary_data["Variable"].append("Rent level (Zillow Rent Index)")
summary_data["Mean"].append(np.nanmean(model_data.business_covariates["ZRI"]))
summary_data["SD"].append(np.nanstd(model_data.business_covariates["ZRI"], ddof=1))

# Restaurant Size
summary_data["Variable"].append("Restaurant Size (in m^2)")
summary_data["Mean"].append(
    np.nanmean(model_data.business_covariates["Restaurant.Size"])
)
summary_data["SD"].append(
    np.nanstd(model_data.business_covariates["Restaurant.Size"], ddof=1)
)

# Number of Seats
summary_data["Variable"].append("Number of Seats")
summary_data["Mean"].append(
    np.nanmean(model_data.business_covariates["Number.of.Seats"])
)
summary_data["SD"].append(
    np.nanstd(model_data.business_covariates["Number.of.Seats"], ddof=1)
)


# Time
summary_data["Variable"].append("Time")
summary_data["Mean"].append(np.mean(model_data.time))
summary_data["SD"].append(np.std(model_data.time, ddof=1))

# Closed
summary_data["Variable"].append("Closed")
summary_data["Mean"].append(np.mean(model_data.closed))
summary_data["SD"].append(np.std(model_data.closed, ddof=1))

# DataFrame erstellen
summary_df = pd.DataFrame(summary_data)
summary_df

,Variable,Mean,SD
0,Rating,3.702020,1.459042
1,Days Between Ratings,19.392615,43.814645
2,Sentiment,0.078033,0.368451
3,Density,12.709012,14.809149
4,Age (in months),48.102528,29.477422
5,Check-in rate,5.601950,6.662546
6,Chain status,0.287731,0.452951
7,Rent level (Zillow Rent Index),1492.515660,179.943743
8,Restaurant Size (in m^2),228.811705,259.853114
9,Number of Seats,98.099462,103.882457


In [4]:
######## Price Levels ###########
price_levels = {"Price Level": [], "Mean": [], "SD": []}

level_to_label = {1: "<10$", 2: "11-30$", 3: "31-60$", 4: ">60$"}

for lvl in range(1, 4 + 1):
    price_levels["Price Level"].append(f"{lvl} ({level_to_label[lvl]})")
    price_levels["Mean"].append(
        np.mean(model_data.business_covariates["Price.Level"] == lvl)
    )
    price_levels["SD"].append(
        np.std(model_data.business_covariates["Price.Level"] == lvl, ddof=1)
    )

price_df = pd.DataFrame(price_levels)
price_df

,Price Level,Mean,SD
0,1 (<10$),0.527687,0.499504
1,2 (11-30$),0.435396,0.496078
2,3 (31-60$),0.008686,0.092845
3,4 (>60$),0.000000,0.000000


In [5]:
############ Restaurant Categories ################

# Finde alle Kategorie-Spalten (beginnen mit "category_")
category_unique = model_data.business_covariates["category"].unique()
# Für jede einzigartige Kategorie berechnen wir den Mean und SD des Auftretens
category_stats = {"Category": [], "Mean": [], "SD": []}

for cat in category_unique:
    indicator = (model_data.business_covariates["category"] == cat).astype(float)
    category_stats["Category"].append(cat)
    category_stats["Mean"].append(np.mean(indicator))
    category_stats["SD"].append(np.std(indicator, ddof=1))

category_df = pd.DataFrame(category_stats)
category_df

,Category,Mean,SD
0,American,0.257329,0.437400
1,Cafes,0.087948,0.283373
2,Other,0.064061,0.244994
3,Mexican,0.174810,0.380011
4,Pizza,0.092291,0.289594
5,Speciality Food,0.054289,0.226710
6,Fast Food,0.108578,0.311278
7,Asian,0.124864,0.330745
8,Salad,0.035831,0.185969


# Laden von trainierten Modellen


In [6]:
from typing import Literal


def load_fitted_model(
    model_name: Literal["hmm", "vdhmm"],
    S: int,
    seed: int = None,
) -> Model:
    """
    Lädt ein trainiertes CmdStanPy Modell.
    """
    model_path = (
        FITTED_MODEL_FOLDER
        / f"{model_name}_{S}{f'_seed{seed}' if seed else ""}_cmdstan.pkl"
    )
    model = Model.from_pickle(model_path)

    return model


# test
hmm_3_seed42_model = load_fitted_model("hmm", 3, seed=42)
print(hmm_3_seed42_model)
print(hmm_3_seed42_model.fit.stan_variables().keys())

Model(model_name='hmm', S=3, n_params=3437)
dict_keys(['pi', 'tpm', 'intercept', 'state_emission_raw', 'probs', 'R2', 'omega_state_raw', 'omega_state_mid', 'omega_tilde', 'omega_0', 'log_lik', 'emission', 'duration_mean', 'duration_sd', 'Duration', 'l_pi', 'l_tpm', 'omega_state', 'close_prob', 'log_lik_test', 'omega_cov'])


In [7]:
model_configurations = [
    ("hmm", 2, 42),
    ("hmm", 3, 42),
    ("hmm", 4, 42),  # Konfigurationen mit Seed 42
    ("vdhmm", 2, 42),
    ("vdhmm", 3, 42),
    ("vdhmm", 4, 42),
    ("hmm", 2, 43),
    ("hmm", 3, 43),
    ("hmm", 4, 43),
    ("vdhmm", 2, 43),
    ("vdhmm", 3, 43),  # Konfigurationen mit Seed 43
    ("vdhmm", 4, 43),
    ("hmm", 2, 123),
    ("hmm", 3, 123),
    ("hmm", 4, 123),  # Konfigrationen mit Seed 123
    ("vdhmm", 2, 123),
    ("vdhmm", 3, 123),
    ("vdhmm", 4, 123),
]

n_models = len(model_configurations)

# get different metrics out of hmm models

results_df = pd.DataFrame(
    {
        "Model": [
            f"{model_name}_{S}" for (model_name, S, seed) in model_configurations
        ],
        "Seed": [seed for (model_name, S, seed) in model_configurations],
        # "LOOIC": [0.0] * n_models,
        # "WAIC": [0.0] * n_models,
        # "LPD_TRAIN": [0.0] * n_models,
        # "LPD_VAL": [0.0] * n_models,
        "AUC_TRAIN": [0.0] * n_models,
        "AUC_TEST": [0.0] * n_models,
        "R_HAT_MEAN": [0.0] * n_models,
        "R_HAT_STD": [0.0] * n_models,
    }
)

In [8]:
import arviz as az
from sklearn.metrics import roc_auc_score
import warnings

# Unterdrücke ArviZ Warnungen und Output
warnings.filterwarnings("ignore")

# Schleife über alle Modell-Konfigurationen
for k, (model_name, S, seed) in enumerate(model_configurations):
    print(f"Processing {k+1}/{n_models}: {model_name} with {S} states (seed={seed})...")

    # Lade das Modell mit Seed
    try:
        model = load_fitted_model(model_name, S, seed=seed)
    except FileNotFoundError:
        print(
            f"FileNotFound: {model_name} with {S} states and seed {seed} was not found! Skipping..."
        )
        continue

    log_lik = model.fit.stan_variable("log_lik")
    log_lik_test = model.fit.stan_variable("log_lik_test")
    close_prob = model.fit.stan_variable("close_prob")
    close_prob_mean = close_prob.mean(axis=0)

    idata = az.from_cmdstanpy(model.fit, log_likelihood="log_lik")

    # Berechne looc, waic, lpd (ohne Output)
    # results_df.loc[k, "LOOIC"] = az.loo(idata, var_name="log_lik").elpd_loo
    # results_df.loc[k, "WAIC"] = az.waic(idata, var_name="log_lik").elpd_waic
    # results_df.loc[k, "LPD_TRAIN"] = -2 * np.sum(np.log(np.exp(log_lik).mean(axis=0)))
    # results_df.loc[k, "LPD_VAL"] = -2 * np.sum(
    #     np.log(np.exp(log_lik_test).mean(axis=0))
    # )

    # Berechne AUC für Trainingsdaten (in Prozent)
    y_true_train = np.array(model.stan_data["Closed"][:500])
    results_df.loc[k, "AUC_TRAIN"] = 100 * roc_auc_score(
        y_true_train, close_prob_mean[:500]
    )

    y_true_test = np.array(model.stan_data["Closed"])[eval_indices]
    # Annahme: Die Schrittweite passt, ggf. muss die Dimensionalität geprüft werden!
    close_prob_test_mean = np.mean(model.fit.stan_variable("close_prob"), axis=0)
    results_df.loc[k, "AUC_TEST"] = 100 * roc_auc_score(
        y_true_test, close_prob_mean[eval_indices]
    )

    # Verwende das bereits gespeicherte summary DataFrame (bereits in model.summary!)
    # Berechne Durchschnitt und Standardabweichung von R_hat über alle Parameter
    results_df.loc[k, "R_HAT_MEAN"] = np.nanmean(model.summary["R_hat"])
    results_df.loc[k, "R_HAT_STD"] = np.nanstd(model.summary["R_hat"])

print("\nErgebnisse:")
results_df

Processing 1/18: hmm with 2 states (seed=42)...
Processing 2/18: hmm with 3 states (seed=42)...
Processing 3/18: hmm with 4 states (seed=42)...
Processing 4/18: vdhmm with 2 states (seed=42)...
Processing 5/18: vdhmm with 3 states (seed=42)...
Processing 6/18: vdhmm with 4 states (seed=42)...
Processing 7/18: hmm with 2 states (seed=43)...
Processing 8/18: hmm with 3 states (seed=43)...
Processing 9/18: hmm with 4 states (seed=43)...
Processing 10/18: vdhmm with 2 states (seed=43)...
Processing 11/18: vdhmm with 3 states (seed=43)...
Processing 12/18: vdhmm with 4 states (seed=43)...
Processing 13/18: hmm with 2 states (seed=123)...
Processing 14/18: hmm with 3 states (seed=123)...
Processing 15/18: hmm with 4 states (seed=123)...
FileNotFound: hmm with 4 states and seed 123 was not found! Skipping...
Processing 16/18: vdhmm with 2 states (seed=123)...
Processing 17/18: vdhmm with 3 states (seed=123)...
Processing 18/18: vdhmm with 4 states (seed=123)...
FileNotFound: vdhmm with 4 stat

,Model,Seed,AUC_TRAIN,AUC_TEST,R_HAT_MEAN,R_HAT_STD
0,hmm_2,42,79.859966,75.591564,1.000592,0.000747
1,hmm_3,42,83.828346,78.960905,1.339409,0.210371
2,hmm_4,42,83.597967,78.616255,1.628896,0.433678
3,vdhmm_2,42,79.259176,75.288066,1.357438,0.229202
4,vdhmm_3,42,85.357425,80.673868,1.000986,0.000901
5,vdhmm_4,42,85.775268,81.250000,1.587418,0.372255
6,hmm_2,43,82.236025,77.021605,1.000583,0.000702
7,hmm_3,43,84.476567,81.409465,1.334442,0.213310
8,hmm_4,43,85.904009,82.422840,1.290499,0.210057
9,vdhmm_2,43,82.848108,74.192387,1.000804,0.000838
